# 02 · BM25: evaluation functions and baseline B1

Runs on **Kaggle CPU**, no internet needed, in about 5–10 minutes.

**Input:** Add Input → Notebook Output → `01_preprocess` (`pairs.parquet` and `products_clean.parquet`).

**Output** (`/kaggle/working`):

| File | Content | In git |
|---|---|:---:|
| `results/<run>.json` | Configuration and metrics on val and test | yes |
| `results/scores/<run>_{val,test}.parquet` | `example_id, query_id, score` | yes |
| `bm25_features.parquet` | `example_id, bm25_title, bm25_main` for **all** splits, used as features in `05_oof_lgbm` | no |

**Runs:** `b0_random` (constant score, i.e. random order), `b1_bm25_title`, `b1_bm25_main`.

**Reference values (from the EDA):** a constant score gives nDCG ≈ 0.742 on val and 0.747 on test; BM25 on titles gives ≈ 0.816 on val.

In [1]:
import glob, json, math, re, time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
TIE_SEEDS = (0, 1, 2)   # ties broken at random, metric averaged over 3 seeds (as in eda/eda_audit.py)
GAINS = {'official': {'E': 1.0, 'S': 0.1, 'C': 0.01, 'I': 0.0},
         'amazon_script': {'E': 1.0, 'S': 0.01, 'C': 0.1, 'I': 0.0}}   # S/C swapped; only to compare with Amazon's 0.857
OUT = Path('/kaggle/working')
RES = OUT / 'results'
(RES / 'scores').mkdir(parents=True, exist_ok=True)


def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not found under /kaggle/input: Add Input -> Notebook Output -> 01_preprocess'
    return Path(hits[0])


pairs = pd.read_parquet(find('pairs.parquet'))
prod = pd.read_parquet(find('products_clean.parquet'), columns=['product_id', 'title', 'brand', 'color', 'bullets'])
print('pairs', pairs.shape, '| products', prod.shape)
print('pairs per split:', pairs['split'].value_counts().to_dict())

pairs (601354, 9) | products (482105, 5)
pairs per split: {'train': 378140, 'test': 181701, 'val': 41513}


## 1. Evaluation functions

The functions in this section will also be used in `03_cross_encoder`; at that point they move to a shared `metrics.py`.

| Metric | Meaning |
|---|---|
| `ndcg` | nDCG over the full candidate list, official gains E 1 / S 0.1 / C 0.01 / I 0. **Main metric.** |
| `ndcg_amazon` | As `ndcg`, with the S and C gains swapped as in Amazon's script. Only for comparison with the 0.857 reference |
| `ndcg10` | nDCG@10, official gains |
| `mrr_e` | 1 / rank of the first E product; only queries with at least one E |
| `e_over_s` | Share of (E, S) pairs within a query where E scores higher; ties count 0.5; pooled over all queries |

- Ties are broken at random, and every metric is the mean over 3 seeds.
- Queries whose candidates are all I have IDCG = 0 and are left out of the means; their number is in `n_skipped_all_irrelevant`.
- 95% confidence intervals come from a bootstrap over queries (1,000 resamples). Two runs on the same queries are compared with a paired bootstrap.

In [2]:
def _positions(codes):
    # 0-based rank inside each query; `codes` must be sorted so every query is contiguous
    n = len(codes)
    start = np.r_[0, np.flatnonzero(np.diff(codes)) + 1]
    return np.arange(n) - np.repeat(start, np.diff(np.r_[start, n]))


def query_metrics(df, score, seeds=TIE_SEEDS):
    # Per-query metrics of one score vector. df needs query_id and esci_label, in the same row order as score.
    score = np.asarray(score, dtype=float)
    assert len(score) == len(df) and np.isfinite(score).all()
    codes, qids = pd.factorize(df['query_id'].to_numpy())
    nq = len(qids)
    lab = df['esci_label'].to_numpy()
    gain = {k: pd.Series(lab).map(m).to_numpy() for k, m in GAINS.items()}
    is_e = lab == 'E'

    def dcg(order, g, k=None):
        c = codes[order]
        pos = _positions(c)
        w = 1 / np.log2(pos + 2)
        if k is not None:
            w = np.where(pos < k, w, 0.0)
        return np.bincount(c, g[order] * w, minlength=nq)

    ideal = {k: np.lexsort((-g, codes)) for k, g in gain.items()}
    idcg = {k: dcg(ideal[k], g) for k, g in gain.items()}
    idcg10 = dcg(ideal['official'], gain['official'], 10)

    acc = {'ndcg': 0.0, 'ndcg_amazon': 0.0, 'ndcg10': 0.0, 'mrr_e': 0.0}
    for sd in seeds:
        order = np.lexsort((np.random.default_rng(sd).random(len(df)), -score, codes))   # query, score desc, random
        acc['ndcg'] = acc['ndcg'] + dcg(order, gain['official'])
        acc['ndcg_amazon'] = acc['ndcg_amazon'] + dcg(order, gain['amazon_script'])
        acc['ndcg10'] = acc['ndcg10'] + dcg(order, gain['official'], 10)
        c, pos, e = codes[order], _positions(codes[order]), is_e[order]
        first_e = np.full(nq, np.inf)
        np.minimum.at(first_e, c[e], pos[e])
        acc['mrr_e'] = acc['mrr_e'] + 1 / (first_e + 1)

    n = len(seeds)
    with np.errstate(divide='ignore', invalid='ignore'):   # IDCG = 0 (all-I query) -> NaN, skipped in means
        return pd.DataFrame({
            'query_id': qids,
            'n_cand': np.bincount(codes, minlength=nq),
            'ndcg': acc['ndcg'] / n / idcg['official'],
            'ndcg_amazon': acc['ndcg_amazon'] / n / idcg['amazon_script'],
            'ndcg10': acc['ndcg10'] / n / idcg10,
            'mrr_e': np.where(np.bincount(codes, is_e.astype(float), minlength=nq) > 0, acc['mrr_e'] / n, np.nan),
        })


def e_over_s(df, score):
    # Share of (E, S) pairs inside a query where E scores higher; ties count 0.5 (Mann-Whitney U), pooled.
    f = pd.DataFrame({'q': df['query_id'].to_numpy(), 'l': df['esci_label'].to_numpy(),
                      's': np.asarray(score, dtype=float)})
    f = f[f['l'].isin(['E', 'S'])].copy()
    f['rk'] = f.groupby('q')['s'].rank(method='average')
    f['is_e'] = f['l'] == 'E'
    agg = f.groupby('q').agg(n_e=('is_e', 'sum'), n=('is_e', 'size'))
    agg['r_e'] = f[f['is_e']].groupby('q')['rk'].sum().reindex(agg.index).fillna(0.0)
    agg['n_s'] = agg['n'] - agg['n_e']
    agg = agg[(agg['n_e'] > 0) & (agg['n_s'] > 0)]
    u = agg['r_e'] - agg['n_e'] * (agg['n_e'] + 1) / 2
    return float(u.sum() / (agg['n_e'] * agg['n_s']).sum())


def bootstrap_ci(x, n_boot=1000, seed=SEED):
    x = np.asarray(x, dtype=float)
    means = x[np.random.default_rng(seed).integers(0, len(x), (n_boot, len(x)))].mean(axis=1)
    return [round(float(np.percentile(means, 2.5)), 4), round(float(np.percentile(means, 97.5)), 4)]


def paired_bootstrap(pq_a, pq_b, metric='ndcg', n_boot=1000, seed=SEED):
    # Mean of (b - a) over the same queries, its 95% CI, and the share of resamples where b <= a.
    d = pq_a[['query_id', metric]].merge(pq_b[['query_id', metric]], on='query_id', suffixes=('_a', '_b')).dropna()
    diff = (d[metric + '_b'] - d[metric + '_a']).to_numpy()
    means = diff[np.random.default_rng(seed).integers(0, len(diff), (n_boot, len(diff)))].mean(axis=1)
    return {'diff': round(float(diff.mean()), 4),
            'ci95': [round(float(np.percentile(means, 2.5)), 4), round(float(np.percentile(means, 97.5)), 4)],
            'p_b_le_a': float((means <= 0).mean())}


METRICS = ['ndcg', 'ndcg_amazon', 'ndcg10', 'mrr_e']


def summarize(df, score):
    pq = query_metrics(df, score)
    s = {m: round(float(pq[m].mean()), 4) for m in METRICS}
    s['ndcg_ci95'] = bootstrap_ci(pq['ndcg'].dropna())
    s['e_over_s'] = round(e_over_s(df, score), 4)
    s['n_queries'] = int(pq['ndcg'].notna().sum())
    s['n_skipped_all_irrelevant'] = int(pq['ndcg'].isna().sum())
    return s, pq

Quick checks of the functions: using the gain itself as the score must give nDCG = 1; the negated gain must score below random; a constant score must give E-over-S = 0.5.

In [3]:
MASK = {s: (pairs['split'] == s).to_numpy() for s in ['val', 'test']}
val = pairs[MASK['val']]

oracle, _ = summarize(val, val['gain'])
worst, _ = summarize(val, -val['gain'])
flat, _ = summarize(val, np.zeros(len(val)))
print('oracle', oracle, '\nworst ', worst, '\nflat  ', flat)
assert oracle['ndcg'] == 1.0 and oracle['ndcg10'] == 1.0 and oracle['mrr_e'] == 1.0 and oracle['e_over_s'] == 1.0
assert worst['ndcg'] < flat['ndcg'] < oracle['ndcg']
assert flat['e_over_s'] == 0.5

oracle {'ndcg': 1.0, 'ndcg_amazon': 0.9983, 'ndcg10': 1.0, 'mrr_e': 1.0, 'ndcg_ci95': [1.0, 1.0], 'e_over_s': 1.0, 'n_queries': 2089, 'n_skipped_all_irrelevant': 0} 
worst  {'ndcg': 0.56, 'ndcg_amazon': 0.5194, 'ndcg10': 0.2056, 'mrr_e': 0.1142, 'ndcg_ci95': [0.554, 0.5656], 'e_over_s': 0.0, 'n_queries': 2089, 'n_skipped_all_irrelevant': 0} 
flat   {'ndcg': 0.7422, 'ndcg_amazon': 0.709, 'ndcg10': 0.5749, 'mrr_e': 0.6063, 'ndcg_ci95': [0.7361, 0.7485], 'e_over_s': 0.5, 'n_queries': 2089, 'n_skipped_all_irrelevant': 0}


## 2. Logging a run

Each run writes `results/<run>.json` and `results/scores/<run>_{val,test}.parquet`. Every notebook that produces scores uses this format, so `06_report` can read them all.

In [4]:
records, per_query = {}, {}


def log_run(name, score_all, info, compare_to=None):
    # score_all: one score per row of `pairs` (all splits, same order)
    score_all = np.asarray(score_all, dtype=float)
    rec = {'run': name, **info, 'tie_seeds': list(TIE_SEEDS), 'bootstrap': 1000,
           'created_utc': time.strftime('%Y-%m-%d %H:%M', time.gmtime()), 'metrics': {}}
    per_query[name] = {}
    for split, m in MASK.items():
        part = pairs[m]
        rec['metrics'][split], per_query[name][split] = summarize(part, score_all[m])
        part[['example_id', 'query_id']].assign(score=score_all[m].astype('float32')).to_parquet(
            RES / 'scores' / f'{name}_{split}.parquet', index=False)
    if compare_to:
        rec['paired_vs'] = {compare_to: {s: paired_bootstrap(per_query[compare_to][s], per_query[name][s])
                                         for s in MASK}}
    records[name] = rec
    (RES / f'{name}.json').write_text(json.dumps(rec, indent=2, ensure_ascii=False))
    print(name, {s: (v['ndcg'], v['ndcg_ci95']) for s, v in rec['metrics'].items()})
    return rec

## 3. B0: random order

Every product gets the same score, so the order is decided entirely by the random tie-break. Every model must beat this floor.

In [5]:
rec = log_run('b0_random', np.zeros(len(pairs)), {'model': 'constant score (random order)'})
assert abs(rec['metrics']['val']['ndcg'] - 0.742) < 0.005 and abs(rec['metrics']['test']['ndcg'] - 0.747) < 0.005

b0_random {'val': (0.7422, [0.7361, 0.7485]), 'test': (0.7459, [0.7429, 0.7491])}


## 4. B1: BM25

- Okapi BM25 with `k1 = 1.2`, `b = 0.75`. IDF and average document length are computed over the **whole catalogue** (482,105 products), not within each candidate list.
- Tokenisation as in the EDA: lowercase, `[a-z0-9]+` tokens, decimals such as `10.2` kept. Each query term counts once.
- Two fields:
  - `title`;
  - `main` = title + brand + color + bullets, joined with spaces. The `text_main` column is not used, because its `brand:` and `color:` labels would become fake terms present in every product.
- BM25 does not learn from labels, so it can score every split and be saved as a LightGBM feature.

In [6]:
TOKEN = re.compile(r'[a-z0-9]+(?:\.[0-9]+)?')


def tokens(s):
    return TOKEN.findall(s.lower()) if isinstance(s, str) else []


class BM25:
    # Okapi BM25; document frequencies and average length come from the whole catalogue.
    def __init__(self, docs, k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        self.df, total = Counter(), 0
        for d in docs:
            t = tokens(d)
            self.df.update(set(t))
            total += len(t)
        self.n = len(docs)
        self.avgdl = total / self.n

    def idf(self, term):
        n = self.df.get(term, 0)
        return math.log(1 + (self.n - n + 0.5) / (n + 0.5))

    def score(self, query, doc):
        dt = tokens(doc)
        if not dt:
            return 0.0
        tf = Counter(dt)
        norm = self.k1 * (1 - self.b + self.b * len(dt) / self.avgdl)
        return sum(self.idf(t) * tf[t] * (self.k1 + 1) / (tf[t] + norm) for t in set(tokens(query)) if t in tf)


FIELDS = {
    'title': prod['title'],
    'main': prod['title'] + ' ' + prod['brand'].fillna('') + ' ' + prod['color'].fillna('') + ' ' + prod['bullets'].fillna(''),
}

feats = pairs[['example_id']].copy()
for field, docs in FIELDS.items():
    t0 = time.time()
    model = BM25(docs.tolist())
    text = pairs['product_id'].map(pd.Series(docs.to_numpy(), index=prod['product_id']))
    feats[f'bm25_{field}'] = [model.score(q, d) for q, d in zip(pairs['query'], text)]
    print(f'{field}: vocab {len(model.df):,}, avgdl {model.avgdl:.1f}, {time.time() - t0:.0f} s')

feats.describe().round(2)

title: vocab 219,469, avgdl 17.3, 15 s
main: vocab 367,953, avgdl 118.9, 70 s


,example_id,bm25_title,bm25_main
count,601354.00,601354.00,601354.00
mean,1095613.19,10.36,13.44
std,678264.10,8.44,10.05
min,16.00,0.00,0.00
25%,498185.25,4.22,6.11
50%,1091791.50,9.44,12.51
75%,1686181.75,15.25,19.32
max,2618569.00,92.26,108.84


In [7]:
info = {'model': 'BM25', 'k1': 1.2, 'b': 0.75, 'idf': 'whole catalogue', 'tokenizer': TOKEN.pattern}
rec = log_run('b1_bm25_title', feats['bm25_title'], {**info, 'fields': 'title'}, compare_to='b0_random')
assert abs(rec['metrics']['val']['ndcg'] - 0.816) < 0.01, 'BM25 title far from the EDA value 0.816'
log_run('b1_bm25_main', feats['bm25_main'], {**info, 'fields': 'title + brand + color + bullets'},
        compare_to='b1_bm25_title');

b1_bm25_title {'val': (0.8162, [0.8096, 0.8227]), 'test': (0.8209, [0.8179, 0.824])}
b1_bm25_main {'val': (0.8227, [0.8165, 0.8293]), 'test': (0.8244, [0.8209, 0.8275])}


## 5. Comparison table

`paired_vs` is the nDCG difference between a run and the run it is compared to, over the same queries. If `ci95` excludes 0, the difference is statistically significant.

In [8]:
cols = ['ndcg', 'ndcg_ci95', 'ndcg_amazon', 'ndcg10', 'mrr_e', 'e_over_s', 'n_queries']
table = pd.DataFrame([{'split': s, 'run': n, **{c: r['metrics'][s][c] for c in cols}}
                      for n, r in records.items() for s in MASK]).set_index(['split', 'run'])
display(table)
for n, r in records.items():
    for ref, by_split in r.get('paired_vs', {}).items():
        for s, v in by_split.items():
            print(f'{s:4s} {n} - {ref}: {v}')

,,ndcg,ndcg_ci95,ndcg_amazon,ndcg10,mrr_e,e_over_s,n_queries
split,run,,,,,,,
val,b0_random,0.7422,"[0.7361, 0.7485]",0.7090,0.5749,0.6063,0.5000,2089
test,b0_random,0.7459,"[0.7429, 0.7491]",0.7133,0.5763,0.6087,0.5000,8956
val,b1_bm25_title,0.8162,"[0.8096, 0.8227]",0.7937,0.6922,0.7465,0.6012,2089
test,b1_bm25_title,0.8209,"[0.8179, 0.824]",0.7996,0.6954,0.7579,0.6103,8956
val,b1_bm25_main,0.8227,"[0.8165, 0.8293]",0.8025,0.7041,0.7568,0.6189,2089
test,b1_bm25_main,0.8244,"[0.8209, 0.8275]",0.8046,0.7012,0.7595,0.6190,8956


val  b1_bm25_title - b0_random: {'diff': 0.074, 'ci95': [0.0672, 0.0798], 'p_b_le_a': 0.0}
test b1_bm25_title - b0_random: {'diff': 0.075, 'ci95': [0.0722, 0.0779], 'p_b_le_a': 0.0}
val  b1_bm25_main - b1_bm25_title: {'diff': 0.0066, 'ci95': [0.0024, 0.0108], 'p_b_le_a': 0.0}
test b1_bm25_main - b1_bm25_title: {'diff': 0.0035, 'ci95': [0.0014, 0.0054], 'p_b_le_a': 0.001}


nDCG on test by query type (`source`), to see where BM25 is strong or weak. For example, queries with negation (`negations`) are hard for BM25, because it rewards the negated term too.

In [9]:
src = pairs.drop_duplicates('query_id').set_index('query_id')['source']
by_src = pd.DataFrame({n: per_query[n]['test'].set_index('query_id')['ndcg'] for n in records})
by_src = by_src.join(src).groupby('source').agg(['mean', 'size']).round(4)
by_src

b0_random       b1_bm25_title       b1_bm25_main      
                   mean  size          mean  size         mean  size
source                                                              
behavioral       0.7570   223        0.8473   223       0.8516   223
negations        0.6979   689        0.7595   689       0.7510   689
nlqec            0.6134    30        0.6453    30       0.7225    30
other            0.7487  7233        0.8235  7233       0.8283  7233
parse_pattern    0.7637   781        0.8500   781       0.8488   781

## 6. Save the BM25 features for all splits

In [10]:
feats.to_parquet(OUT / 'bm25_features.parquet', index=False)
for f in sorted(p for p in OUT.rglob('*') if p.is_file()):
    print(f'{str(f.relative_to(OUT)):42s} {f.stat().st_size / 1e6:7.2f} MB')

__notebook__.ipynb                            0.04 MB
bm25_features.parquet                        10.83 MB
results/b0_random.json                        0.00 MB
results/b1_bm25_main.json                     0.00 MB
results/b1_bm25_title.json                    0.00 MB
results/scores/b0_random_test.parquet         1.07 MB
results/scores/b0_random_val.parquet          0.27 MB
results/scores/b1_bm25_main_test.parquet      2.03 MB
results/scores/b1_bm25_main_val.parquet       0.48 MB
results/scores/b1_bm25_title_test.parquet     1.83 MB
results/scores/b1_bm25_title_val.parquet      0.44 MB
